<a href="https://colab.research.google.com/github/sophiabratteson/long-short-equity-screener/blob/main/screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import yfinance as yf
import pandas as pd

tickers = ["GOOGL", "META", "PINS", "SNAP", "RDDT", "MTCH", "YELP"]
metrics = ["shortName", "marketCap", "trailingPE", "forwardPE",
           "profitMargins", "revenueGrowth", "debtToEquity", "returnOnEquity"]

rows = []                          # empty list; one entry per company
for t in tickers:                  # t becomes "GOOGL", then "META", ...
    info = yf.Ticker(t).info
    row = {"ticker": t}            # a dictionary for this one company
    for m in metrics:              # loop inside a loop: one pass per metric
        row[m] = info.get(m)
    rows.append(row)               # add the metric for ticker to list rows

df = pd.DataFrame(rows).set_index("ticker")   # turn the list into a table
df

,shortName,marketCap,trailingPE,forwardPE,profitMargins,revenueGrowth,debtToEquity,returnOnEquity
ticker,,,,,,,,
GOOGL,Alphabet Inc.,3533049692160,15.006789,18.618591,0.50031,0.247,22.187,0.44286
META,"Meta Platforms, Inc.",1639927119872,21.882977,18.022726,0.31145,0.261,42.161,0.32264
PINS,"Pinterest, Inc.",10867974144,78.958336,8.325645,0.03810,0.119,38.538,0.04498
SNAP,Snap Inc.,8632884224,NaN,6.356229,-0.02303,0.143,185.090,-0.06784
RDDT,"Reddit, Inc.",19052558336,20.136734,10.625958,0.32105,0.530,0.877,0.32030
MTCH,"Match Group, Inc.",9087526912,13.402684,9.968801,0.20692,-0.025,NaN,NaN
YELP,Yelp Inc.,1098031872,9.641149,5.510809,0.08293,-0.014,14.372,0.17328


In [21]:
df.isna().sum()      # counts the blank (missing) cells in each column
df.describe()        # min, max, average and spread of every number column

,marketCap,trailingPE,forwardPE,profitMargins,revenueGrowth,debtToEquity,returnOnEquity
count,7.000000e+00,6.000000,7.000000,7.000000,7.000000,6.000000,6.000000
mean,7.459594e+11,26.504778,11.061251,0.205390,0.180143,50.537500,0.206037
std,1.370964e+12,26.082712,5.281834,0.185636,0.190745,67.670361,0.192228
min,1.098032e+09,9.641149,5.510809,-0.023030,-0.025000,0.877000,-0.067840
25%,8.860206e+09,13.803710,7.340937,0.060515,0.052500,16.325750,0.077055
50%,1.086797e+10,17.571762,9.968801,0.206920,0.143000,30.362500,0.246790
75%,8.294898e+11,21.446416,14.324342,0.316250,0.254000,41.255250,0.322055
max,3.533050e+12,78.958336,18.618591,0.500310,0.530000,185.090000,0.442860


In [28]:
df["earnings_yield"] = 1/df["forwardPE"]
df.describe()

,marketCap,trailingPE,forwardPE,profitMargins,revenueGrowth,debtToEquity,returnOnEquity,earnings_yield,earnings_yield_z,revenueGrowth_z,profitMargins_z,debtToEquity_z,score
count,7.000000e+00,6.000000,7.000000,7.000000,7.000000,6.000000,6.000000,7.000000,7.000000e+00,7.000000e+00,7.000000e+00,6.000000e+00,7.000000
mean,7.459594e+11,26.504778,11.061251,0.205390,0.180143,50.537500,0.206037,0.108931,9.516197e-17,-9.516197e-17,1.268826e-16,1.850372e-17,-0.014835
std,1.370964e+12,26.082712,5.281834,0.185636,0.190745,67.670361,0.192228,0.048179,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.446211
min,1.098032e+09,9.641149,5.510809,-0.023030,-0.025000,0.877000,-0.067840,0.053710,-1.146156e+00,-1.075485e+00,-1.230473e+00,-1.988352e+00,-0.602268
25%,8.860206e+09,13.803710,7.340937,0.060515,0.052500,16.325750,0.077055,0.074797,-7.084669e-01,-6.691823e-01,-7.804254e-01,1.371686e-01,-0.309229
50%,1.086797e+10,17.571762,9.968801,0.206920,0.143000,30.362500,0.246790,0.100313,-1.788702e-01,-1.947257e-01,8.241939e-03,2.981364e-01,0.002430
75%,8.294898e+11,21.446416,14.324342,0.316250,0.254000,41.255250,0.322055,0.138718,6.182643e-01,3.872045e-01,5.971904e-01,5.055648e-01,0.196797
max,3.533050e+12,78.958336,18.618591,0.500310,0.530000,185.090000,0.442860,0.181462,1.505431e+00,1.834166e+00,1.588701e+00,7.338590e-01,0.720859


In [27]:
# 1 = higher is better, -1 = lower is better
factors = {"earnings_yield": 1, "revenueGrowth": 1,
           "profitMargins": 1, "debtToEquity": -1}

for col, direction in factors.items():   # col = column name, direction = 1 or -1
    z = (df[col] - df[col].mean()) / df[col].std()
    df[col + "_z"] = z * direction

In [26]:
z_cols = ["earnings_yield_z", "revenueGrowth_z", "profitMargins_z", "debtToEquity_z"]
df["score"] = df[z_cols].mean(axis=1)
df.sort_values("score", ascending=False)[["shortName", "score"] + z_cols]

,shortName,score,earnings_yield_z,revenueGrowth_z,profitMargins_z,debtToEquity_z
ticker,,,,,,
RDDT,"Reddit, Inc.",0.720859,-0.307635,1.834166,0.623047,0.733859
GOOGL,Alphabet Inc.,0.303000,-1.146156,0.350506,1.588701,0.418950
YELP,Yelp Inc.,0.090593,1.505431,-1.017816,-0.659678,0.534436
META,"Meta Platforms, Inc.",0.002430,-1.109299,0.423903,0.571333,0.123784
PINS,"Pinterest, Inc.",-0.203087,0.232049,-0.320548,-0.901173,0.177323
MTCH,"Match Group, Inc.",-0.415371,-0.178870,-1.075485,0.008242,NaN
SNAP,Snap Inc.,-0.602268,1.004479,-0.194726,-1.230473,-1.988352
